# Session 1: Framing and EDA
**Emission-trajectory ML project. Dr. Khawar Naeem, QTTSC, Qatar University.**

Goal of this notebook: load the OWID CO2 dataset, understand its structure, separate real countries from aggregates, and state the prediction problem precisely. No modeling here.

**The core question:** using information available through year *t*, how accurately can we predict a country's production-based CO2 emissions in year *t+1*, and does ML beat simple baselines?

Run each cell, read the explanation above it, and answer the check questions at the end.

## 1. Load the data

**Concept: panel data.** This is not a plain table of independent rows. It is *panel data*: repeated yearly observations of the same entities (countries). Almost every mistake in this project (leakage, bad splits, broken lags) comes from forgetting that rows belong to country time-series.

**Why pandas here is the efficient approach:** the CSV is 14 MB, about 50k rows. That fits in memory thousands of times over, so a database adds no value at this stage; `read_csv` gives us one in-memory table we can group by country and manipulate vectorized (no Python loops over 218 countries).

In [ ]:
import pandas as pd

pd.set_option("display.max_columns", 100)

df = pd.read_csv("../data/raw/owid-co2-data.csv")
print(df.shape)          # expect roughly (50411, 79)
print(df["year"].min(), "-", df["year"].max())   # 1750 - 2024
df.head(3)

## 2. What the columns mean

79 columns, but they fall into a few families (full definitions are in `../data/raw/owid-co2-codebook.csv`):

| Family | Examples | Note |
|---|---|---|
| Identity | `country`, `iso_code`, `year` | the panel keys |
| Target family | `co2` (Mt, production-based, territorial) | **`co2` is our target's source column** |
| Per-capita / intensity | `co2_per_capita`, `co2_per_gdp` | |
| Fuel decomposition | `coal_co2`, `oil_co2`, `gas_co2`, `cement_co2`, `flaring_co2` | ~38-44% missing for coal/gas |
| Context | `population`, `gdp`, `primary_energy_consumption` | gdp ~29% missing post-1990 |
| Derived | `share_global_co2`, `co2_growth_prct`, `cumulative_co2` | careful: growth columns describe year *t* itself |
| Other gases / accounting | `methane`, `consumption_co2`, `trade_co2` | consumption_co2 ~47% missing |

**Important:** `co2` here means annual territorial production-based CO2 in million tonnes. It is not consumption-based, not all greenhouse gases, and our model of it is a statistical extrapolation, not a policy forecast.

In [ ]:
codebook = pd.read_csv("../data/raw/owid-co2-codebook.csv")
# Look up the exact definition of the target source column
codebook[codebook["column"].isin(["co2", "co2_per_capita", "share_global_co2"])][["column", "description"]]

## 3. Separate countries from aggregates

**Why this matters:** OWID includes entities like `World`, `Asia`, `European Union (27)`. If they stay in, we double-count (China is inside Asia which is inside World) and the model trains on fictional 'countries' with enormous emissions.

**The rule (from the codebook conventions):** real countries have a 3-letter ISO code; aggregates have either no `iso_code` or one starting with `OWID_` (e.g. `OWID_WRL` for World). Expect about 218 countries and 36 aggregates.

In [ ]:
is_country = df["iso_code"].notna() & ~df["iso_code"].astype(str).str.startswith("OWID")

countries = df[is_country].copy()
aggregates = df[~is_country]

print("country entities:  ", countries["country"].nunique())
print("aggregate entities:", aggregates["country"].nunique())
print()
print("aggregates kept OUT of the model:")
print(sorted(aggregates["country"].unique()))

## 4. Coverage and missingness

Two questions decide our eligibility rules and feature choices:
1. How much history does each country have? (we require at least 40 usable years)
2. Which columns are complete enough to be first-wave features?

**Finding to verify below (from the 8 July inspection):** `co2` and `population` are nearly complete after 1990; `coal_co2`/`gas_co2`/`gdp`/`consumption_co2` have 29-47% missing. So the first modeling table leans on `co2` lags/rolls, `population`, `co2_per_capita`, `share_global_co2`; fuel mix and GDP come later with explicit missing-data handling.

In [ ]:
modern = countries[countries["year"] >= 1990]
key_cols = ["co2", "co2_per_capita", "coal_co2", "oil_co2", "gas_co2", "cement_co2",
            "flaring_co2", "population", "gdp", "share_global_co2", "consumption_co2"]
missing = modern[key_cols].isna().mean().mul(100).round(1).sort_values()
print("% missing in country rows, 1990+:")
print(missing)

obs = countries.dropna(subset=["co2"]).groupby("country").size()
print(f"\ncountries with >= 40 years of co2 data: {(obs >= 40).sum()} of {obs.size}")

## 5. The scale problem

Country emissions span five orders of magnitude. This has a direct consequence for evaluation: an overall MAE is dominated by China and the US. That is why the plan (Amendment 3 in `docs/ML_Learning_Plan_KhawarNaeem.md`) requires median errors, percentage errors, and size-tier breakdowns alongside MAE/RMSE.

In [ ]:
lvl = countries[(countries["year"] == 2023) & countries["co2"].notna()].set_index("country")["co2"]
top5 = lvl.nlargest(5)
print(top5)
print(f"\ntop-5 share of world total: {top5.sum() / lvl.sum() * 100:.0f}%")
lvl.describe()

## 6. Why persistence will be hard to beat

Persistence says: prediction for t+1 = value at t. Emissions change slowly, so this 'dumb' baseline is strong. Verify: the median absolute year-over-year change since 2010 is only ~4-5% of the level.

**Note the pattern below:** `groupby("country")["co2"].diff()`. The groupby is what stops Zimbabwe's first year being differenced against Zambia's last year. Every lag and rolling feature in Session 2 uses this same discipline.

In [ ]:
c = countries.sort_values(["country", "year"]).copy()
c["yoy_change"] = c.groupby("country")["co2"].diff()   # groupby prevents cross-country leakage

recent = c[(c["year"] >= 2010) & c["yoy_change"].notna() & (c["co2"] > 0)]
median_rel_change = (recent["yoy_change"].abs() / recent["co2"]).median() * 100
print(f"median |YoY change| as % of level, 2010+: {median_rel_change:.2f}%")

## 7. Sanity anchor: Qatar

Always keep one country you know well as a manual check. Qatar's emissions grew every year 2020-2024; in Session 2 we verify by hand that Qatar's 2023 row has target = its 2024 `co2` value.

In [ ]:
countries[(countries["country"] == "Qatar") & (countries["year"] >= 2015)][
    ["year", "co2", "co2_per_capita", "population", "share_global_co2"]]

## 8. Problem statement (frozen)

- **Unit of observation:** one country-year row.
- **Target:** `target = co2[t+1]`, i.e. next year's production-based CO2 in Mt, built with a per-country shift.
- **Features for predicting year t+1:** may use data up to and including year *t* only. Nothing later, ever.
- **Eligibility:** ISO-coded countries, no `OWID_` codes, >= 40 usable years, population > 1M in the prediction year. Every exclusion documented in the README.
- **Split (chronological, never random):** train <= 2014, validation 2015-2018, test 2019-2023. 2024 targets are excluded from the headline test set because the newest OWID release year is provisional and subject to revision; 2024 predictions are reported separately in a clearly labeled provisional appendix (decided 8 Jul 2026). Features from 2023 and earlier are settled, so only the target side is affected.
- **Baselines:** persistence and recent linear trend, reported beside every model with a skill score: `skill = 1 - MAE_model / MAE_persistence`.
- **Models may predict the delta** `co2[t+1] - co2[t]` internally, but every model is scored after converting back to the level, on identical rows.

## Check questions (answer before closing the session)

1. Why must `Asia` and `European Union (27)` rows be excluded from training?
2. A feature for year *t* accidentally uses `co2_growth_prct` from year *t+1*. What is this error called, and why would it make validation results look great and real forecasts fail?
3. Why is the validation set 2015-2018 rather than a random 20% of rows?
4. Why is persistence expected to be a strong baseline for this particular variable?
5. Whose row count is bigger in this dataset, `World` or `China`, and why does it not matter for our model?

Answers are recorded in `01_framing_eda_check_questions.md` (8 Jul 2026).

**End-of-session protocol:** save this notebook, update the status table and session log in `CLAUDE.md`, record the exact next action (Session 2: build the modeling table in `src/build_features.py`).